In [4]:
from bs4 import BeautifulSoup
import re
import tiktoken

# Load HTML (normally from a file or variable; here you paste your string)
with open("index.html", "r", encoding="utf-8") as f:
    html_content = f.read()

soup = BeautifulSoup(html_content, "html.parser")
articles = soup.find_all("li", class_="article-item")

enc = tiktoken.encoding_for_model("gpt-3.5-turbo")

title_token_count = 0
title_word_count = 0
abstract_token_count = 0
abstract_word_count = 0

for art in articles:
    title = art.get("data-title", "").strip()
    abstract = art.get("data-abstract", "").strip()

    # Count words
    title_word_count += len(title.split())
    abstract_word_count += len(abstract.split())

    # Count tokens
    title_token_count += len(enc.encode(title))
    abstract_token_count += len(enc.encode(abstract))

summary = {
    "Total Articles": len(articles),
    "Title Word Count": title_word_count,
    "Title Token Count": title_token_count,
    "Abstract Word Count": abstract_word_count,
    "Abstract Token Count": abstract_token_count,
}

summary


{'Total Articles': 7424,
 'Title Word Count': 93924,
 'Title Token Count': 144468,
 'Abstract Word Count': 634392,
 'Abstract Token Count': 915946}

In [23]:
import xml.etree.ElementTree as ET

class TOCmanager:
    def __init__(self, file='all_journals_toc.xml'):
        self.file = file
        self.tree = ET.parse(file)
        self.root = self.tree.getroot()
        self.journal2index = {j.attrib['name']: i for i, j in enumerate(self.root)}
        self.journalsList = list(self.journal2index.keys())
        self.journal_tot = [len(j) for j in self.root]
        self.tot = sum(self.journal_tot)
        self.reverse_gaids = dict()
        self.gaids = dict()
        gaid = 0
        for jid, tot in enumerate(self.journal_tot):
            for aid in range(tot):
                self.gaids[gaid] = (jid, aid)
                self.reverse_gaids[(jid, aid)] = gaid
                gaid += 1

    def get(self, jid=0, aid=0):
        if aid >= self.journal_tot[jid]:
            print(f'Journal {self.journalsList[jid]} (JID:{jid}) has {self.journal_tot[jid]} articles. AID requested: {aid}')
            return None
        article = {elem.tag: elem.text for elem in self.root[jid][aid]}
        if len(article.get('Abstract', '') or '') < 100:
            article['Abstract'] = ''
        article['Journal'] = self.journalsList[jid]
        article['jid'] = jid
        article['aid'] = aid
        article['gaid'] = self.reverse_gaids[(jid, aid)]
        return article

    def gaid(self, gaid):
        jid, aid = self.gaids[gaid]
        return self.get(jid, aid)

    def gaid_batch(self, gaids):
        return [self.get(*self.gaids[gaid]) for gaid in gaids]

    def add_field(self, jid, aid, field_name, value):
        """Add or update a field (e.g. keywords) in a specific article."""
        article_elem = self.root[jid][aid]
        existing = article_elem.find(field_name)
        if existing is not None:
            existing.text = value
        else:
            new_elem = ET.SubElement(article_elem, field_name)
            new_elem.text = value

    def save(self, path=None):
        """Write changes back to file."""
        self.tree.write(path or self.file, encoding='utf-8', xml_declaration=True)

    def print(self, article):
        print(f"### Article - JID: {article['jid']} AID: {article['aid']} - GAID: {article['gaid']} ###")
        for k, t in article.items():
            if k in ('aid', 'jid', 'gaid'):
                continue
            print(f"\t{k} : {t}")

    def info(self):
        print("### TOCS ###")
        print(f"\tJournals: {len(self.journal_tot)}")
        print(f"\tArticles: {self.tot}")

    def __str__(self):
        self.info()
        return ''

    def add_field_gaid(self, gaid, field_name, value):
        """Add or update a field in an article given its global article ID (gaid)."""
        if gaid not in self.gaids:
            print(f"Invalid GAID: {gaid}")
            return
        jid, aid = self.gaids[gaid]
        self.add_field(jid, aid, field_name, value)

    def delete_article(self, jid, aid):
        """Delete one article by journal and article index."""
        if jid >= len(self.root):
            print(f"Invalid journal ID: {jid}")
            return
        if aid >= len(self.root[jid]):
            print(f"Journal {self.journalsList[jid]} has only {len(self.root[jid])} articles.")
            return
        # Remove the element
        del self.root[jid][aid]
        # Update bookkeeping
        self.journal_tot[jid] = len(self.root[jid])
        self.tot = sum(self.journal_tot)
        # Rebuild GAID mappings
        self.gaids.clear()
        self.reverse_gaids.clear()
        gaid = 0
        for j, tot in enumerate(self.journal_tot):
            for a in range(tot):
                self.gaids[gaid] = (j, a)
                self.reverse_gaids[(j, a)] = gaid
                gaid += 1
        print(f"Deleted article {aid} from journal {self.journalsList[jid]}")

    def delete_article_gaid(self, gaid):
        """Delete one article using its global article ID (GAID)."""
        if gaid not in self.gaids:
            print(f"Invalid GAID: {gaid}")
            return
        jid, aid = self.gaids[gaid]
        self.delete_article(jid, aid)

    def find_by_doi(self, doi):
        """Return (jid, aid) of an article with the given DOI."""
        for jid, journal in enumerate(self.root):
            for aid, article in enumerate(journal):
                doi_elem = article.find('DOI')
                if doi_elem is not None and doi_elem.text == doi:
                    return jid, aid
        return None

    def delete_by_doi(self, doi):
        """Delete an article by its DOI."""
        res = self.find_by_doi(doi)
        if not res:
            print(f"DOI not found: {doi}")
            return
        jid, aid = res
        self.delete_article(jid, aid)
        print(f"Deleted article with DOI {doi}")

    def add_field_doi(self, doi, field_name, value):
        """Add or update a field in an article identified by its DOI."""
        res = self.find_by_doi(doi)
        if not res:
            print(f"DOI not found: {doi}")
            return
        jid, aid = res
        self.add_field(jid, aid, field_name, value)



import random

tocs = TOCmanager()
tocs.info()
#article = tocs.gaid(10)
#tocs.print(article)
gaids = random.sample(range(tocs.tot), 20)
articles = tocs.gaid_batch(gaids)
prepared = [(article['Title'],article['Abstract']) for article in articles]
prepared

### TOCS ###
	Journals: 47
	Articles: 3712


[('Mechanical control of cell fate decisions in the skin epidermis', ''),
 ('Embryonic development: Tracing the origin of epiblast cells', ''),
 ('Witness stress promotes age and sex-dependent behavioral and neurofunctional alterations in the amygdaloid complex and dorsal hippocampus in mice',
  ''),
 ('Wired for growth: neuron–tumour signalling in the lung and brain increases growth of a hard-to-treat cancer',
  ''),
 ('A sensory and motor neuropathy caused by a genetic variant of\n            <i>NAMPT</i>',
  'Nicotinamide phosphoribosyl transferase (NAMPT) is the rate-limiting enzyme in the salvage pathway for nicotinamide adenine dinucleotide (NAD+) biosynthesis in mammalian cells and is essential for survival. Here, we report on a previously unidentified axonal sensory and motor neuropathy likely caused by a homozygous genetic variant of missense mutation (c.472G>C, p.P158A) in theNAMPTgene. Two affected siblings presented with a range of clinical features including impaired motor

In [24]:
articles

[{'Title': 'Mechanical control of cell fate decisions in the skin epidermis',
  'Type': 'journal-article',
  'PublicationDate': '2025/9',
  'Authors': 'Sahu P.; Monteiro-Ferreira S.; Canato S.; Soares R.; Sánchez-Danés A.; Hannezo E.',
  'DOI': 'https://doi.org/10.1038/s41467-025-62882-9',
  'Abstract': '',
  'Journal': 'Nature Communications',
  'jid': 10,
  'aid': 53,
  'gaid': 874},
 {'Title': 'Embryonic development: Tracing the origin of epiblast cells',
  'Type': 'journal-article',
  'PublicationDate': '2025/9',
  'Authors': 'Paulus V.; Chazaud C.',
  'DOI': 'https://doi.org/10.1016/j.cub.2025.08.018',
  'Abstract': '',
  'Journal': 'Current Biology',
  'jid': 20,
  'aid': 99,
  'gaid': 1967},
 {'Title': 'Witness stress promotes age and sex-dependent behavioral and neurofunctional alterations in the amygdaloid complex and dorsal hippocampus in mice',
  'Type': 'journal-article',
  'PublicationDate': '2025/10',
  'Authors': 'Avalo-Zuluaga J.; Viatela Ramírez S.; Baptista-de-Souza D

In [20]:
from falcon_tests import ArticleClassifier2


In [21]:
ac = ArticleClassifier2()
ac.load_keywords('keywords.json')

In [22]:
for i,art in enumerate(prepared):
    print(f'### Article {i} ###')
    out = ac.classify(art,parse=True)
    print('\t',art[0])
    print('\t',out)


### Article 0 ###
	 Regulation of Glypican 6-mediated Wnt activation maintains TDP-43 nuclear localization in neurons
	 {'neuroscience': 'yes', 'type': 'article', 'generic_keywords': ['neurodegenerative diseases', 'nucleocytoplasmic trafficking', 'nuclear pore complex'], 'specific_keywords': ['Glypican 6', 'canonical Wnt signaling', 'TDP-43 mislocalization']}
### Article 1 ###
	 A tale of tumors
	 {'neuroscience': 'no', 'type': 'other', 'generic_keywords': ['neurooncology'], 'specific_keywords': ['-']}
### Article 2 ###


KeyboardInterrupt: 

In [5]:
ac.known_keywords

{'Bayesian',
 'DREADDs',
 'GPU',
 'NG100',
 'NextBrain',
 'Nogo inhibition',
 'PVN',
 'analgesia',
 'antibiotics',
 'behavioral testing',
 'biomarkers',
 'brain MRI',
 'cardioprotection',
 'clinical trial',
 'constipation',
 'depression',
 'diffeomorphic registration',
 'electrophysiology',
 'excitatory neurons',
 'fibrosis',
 'gut dysbiosis',
 'gut microbiome',
 'heart disease',
 'histological atlas',
 'hypothalamus',
 'kurkinol',
 'kurkinorin',
 'medial prefrontal cortex',
 'metabolic syndrome',
 'morphine',
 'morphine withdrawal',
 'neurorehabilitation',
 'opioid receptor',
 'oxytocin',
 'prefrontal cortex',
 'recovery',
 'respiratory depression',
 'segmentation',
 'side effects',
 'sleep deprivation',
 'somatosensory evoked potentials',
 'spinal cord injury',
 'synaptic plasticity',
 'tolerance',
 'vasoactive intestinal peptide',
 'β-Arrestin 2'}

In [20]:
!ollama serve

Error: listen tcp 127.0.0.1:11434: bind: address already in use


In [23]:
!ps aux | grep ollama

morning+   61703  0.0  0.0 1930512 31880 pts/2   Tl   22:34   0:00 ollama run mistral
ollama     64555  0.8  0.0 2004240 32164 ?       Ssl  22:51   0:00 /usr/local/bin/ollama serve
morning+   64581 83.3  0.0 231928  3720 pts/3    Ss+  22:51   0:00 /bin/bash -c ps aux | grep ollama
morning+   64583  0.0  0.0 231252  2392 pts/3    S+   22:51   0:00 grep ollama


In [18]:
!sudo pkill ollama

[sudo] password for morningrise: 
